In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

# Cargar las variables del archivo .env (API Keys y configuración)
load_dotenv()

LLM_API_KEY = os.getenv("LLM_API_KEY")
LLM_BASE_URL = "https://api.mistral.ai/v1"
LLM_MODEL = "ministral-8b-latest"
EMBEDDING_MODEL = "mistral-embed"

# Configuración para habilitar la trazabilidad en LangSmith (IL1.4)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "SP-Digital-RAG-Evaluation")

assert LLM_API_KEY, "Error: No se encontró la variable LLM_API_KEY en el archivo .env"

# Creamos el cliente de OpenAI apuntando a Mistral y envuelto para LangSmith
client = wrap_openai(OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL))
print("Cliente conectado a Mistral y trazabilidad de LangSmith activada correctamente.")

Cliente listo y trazabilidad LangSmith activada.


In [ ]:
# Catálogo de productos de prueba para SP Digital
CATALOGO = [
    {
        "sku": "GPU-4070-ASUS",
        "nombre": "ASUS Dual RTX 4070 OC 12GB",
        "categoria": "GPU",
        "marca": "ASUS",
        "specs": "GPU NVIDIA GeForce RTX 4070, 12GB GDDR6X, conector 1x 16-pin (12VHPWR), consumo tipico 200W, fuente recomendada: 650W minimo, PCIe 4.0 x16."
    },
    {
        "sku": "MB-B450-MSI",
        "nombre": "MSI B450 Tomahawk Max",
        "categoria": "Placa madre",
        "marca": "MSI",
        "specs": "Placa madre socket AM4, chipset B450, soporta Ryzen serie 1000 a 5000, 1x slot PCIe 3.0 x16, soporta memoria DDR4."
    },
    {
        "sku": "MB-B650-ASUS",
        "nombre": "ASUS TUF Gaming B650-PLUS WiFi",
        "categoria": "Placa madre",
        "marca": "ASUS",
        "specs": "Placa madre socket AM5, chipset B650, soporta Ryzen serie 7000/8000/9000, 1x slot PCIe 4.0 x16, soporta solo memoria DDR5."
    },
    {
        "sku": "CPU-R5-5600X",
        "nombre": "AMD Ryzen 5 5600X",
        "categoria": "Procesador",
        "marca": "AMD",
        "specs": "Procesador Socket AM4, 6 nucleos, 12 hilos, TDP 65W, no incluye graficos integrados, compatible con memoria DDR4."
    },
    {
        "sku": "CPU-R7-7800X3D",
        "nombre": "AMD Ryzen 7 7800X3D",
        "categoria": "Procesador",
        "marca": "AMD",
        "specs": "Procesador Socket AM5, 8 nucleos, 16 hilos, TDP 120W, requiere cooler, compatible con memoria DDR5."
    },
    {
        "sku": "RAM-DDR4-16GB-CORSAIR",
        "nombre": "Corsair Vengeance LPX 16GB (2x8GB) DDR4 3200MHz",
        "categoria": "Memoria RAM",
        "marca": "Corsair",
        "specs": "Memoria RAM DDR4, capacidad 16GB (2x8GB), frecuencia 3200MHz, formato UDIMM para PC de escritorio."
    },
    {
        "sku": "RAM-DDR5-32GB-KINGSTON",
        "nombre": "Kingston Fury Beast 32GB (2x16GB) DDR5 5600MHz",
        "categoria": "Memoria RAM",
        "marca": "Kingston",
        "specs": "Memoria RAM DDR5, capacidad 32GB (2x16GB), frecuencia 5600MHz, formato UDIMM para PC de escritorio."
    },
    {
        "sku": "PSU-CORSAIR-650",
        "nombre": "Corsair RM650x 650W 80+ Gold",
        "categoria": "Fuente de poder",
        "marca": "Corsair",
        "specs": "Fuente 650W, 80+ Gold, modular, 1x conector 12VHPWR (con adaptador), 2x conectores PCIe 8-pin."
    },
    {
        "sku": "PSU-EVGA-450",
        "nombre": "EVGA 450 BR 450W 80+ Bronze",
        "categoria": "Fuente de poder",
        "marca": "EVGA",
        "specs": "Fuente 450W, 80+ Bronze, no modular, 2x conectores PCIe 8-pin (6+2). No posee conector 12VHPWR."
    }
]

# Reglas duras de compatibilidad para validar las respuestas del LLM
REGLAS_COMPATIBILIDAD = {
    ("GPU-4070-ASUS", "MB-B450-MSI"): {
        "compatible": True,
        "motivo": "El slot PCIe 3.0 x16 acepta la GPU (PCIe 4.0 es retrocompatible).",
        "requisito_adicional": "Fuente >= 650W con conector 12VHPWR o adaptador."
    },
    ("CPU-R7-7800X3D", "MB-B450-MSI"): {
        "compatible": False,
        "motivo": "Incompatibilidad de socket: El procesador es AM5 y la placa madre es AM4.",
        "requisito_adicional": "Requiere una placa madre con Socket AM5 (ej. B650/X670)."
    },
    ("RAM-DDR5-32GB-KINGSTON", "MB-B450-MSI"): {
        "compatible": False,
        "motivo": "Incompatibilidad de memoria: La placa MSI B450 solo soporta DDR4, no DDR5.",
        "requisito_adicional": "Usar memoria DDR4 o cambiar a placa AM5/Intel compatible con DDR5."
    },
    ("GPU-4070-ASUS", "PSU-EVGA-450"): {
        "compatible": False,
        "motivo": "Potencia insuficiente: La GPU requiere minimo 650W y la fuente es de 450W.",
        "requisito_adicional": "Cambiar fuente a minimo 650W 80+ Gold con soporte 12VHPWR."
    }
}

# Documentos técnicos adicionales para alimentar el contexto del RAG
NOTAS_TECNICAS = [
    {
        "sku": "NOTA-PCIE-RETROCOMPAT",
        "nombre": "Nota tecnica: retrocompatibilidad PCIe",
        "categoria": "Nota tecnica",
        "marca": "N/A",
        "specs": "Los slots y tarjetas PCI Express (PCIe) son retrocompatibles entre "
                 "generaciones: una tarjeta PCIe 4.0 funciona correctamente en un slot "
                 "PCIe 3.0, y viceversa. La diferencia de generacion no impide el "
                 "funcionamiento, solo limita el ancho de banda maximo teorico; en la "
                 "practica esto casi no afecta el rendimiento de una GPU en uso gamer."
    }
]

print(f"Cargados: {len(CATALOGO)} productos y {len(NOTAS_TECNICAS)} notas técnicas.")

9 productos y 1 notas técnicas cargadas


In [ ]:
# Función para convertir los datos en bloques de texto (chunks) para el RAG
def construir_chunks(catalogo):
    chunks = []
    for p in catalogo:
        texto = f"Producto: {p['nombre']} (SKU: {p['sku']}). Categoria: {p['categoria']}. " \
                f"Marca: {p['marca']}. Especificaciones: {p['specs']}"
        chunks.append({"texto": texto, "metadata": {"sku": p["sku"]}})
    return chunks

CHUNKS = construir_chunks(CATALOGO + NOTAS_TECNICAS)
print("SKUs procesados para vectorización:")
for c in CHUNKS:
    print("-", c["metadata"]["sku"])

GPU-4070-ASUS
MB-B450-MSI
MB-B650-ASUS
CPU-R5-5600X
CPU-R7-7800X3D
RAM-DDR4-16GB-CORSAIR
RAM-DDR5-32GB-KINGSTON
PSU-CORSAIR-650
PSU-EVGA-450
NOTA-PCIE-RETROCOMPAT


In [ ]:
import numpy as np
import faiss

# Función para sacar los vectores embeddings de los textos con Mistral
def obtener_embeddings(textos):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=textos)
    return np.array([d.embedding for d in resp.data], dtype="float32")

# Creamos el índice de búsqueda vectorial FAISS
embeddings_catalogo = obtener_embeddings([c["texto"] for c in CHUNKS])
faiss.normalize_L2(embeddings_catalogo)
index = faiss.IndexFlatIP(embeddings_catalogo.shape[1])
index.add(embeddings_catalogo)

print("Índice FAISS listo con", index.ntotal, "vectores cargados.")

Indice listo: 10 vectores


In [ ]:
# Módulo de recuperación RAG (retriever) monitoreado por LangSmith
@traceable(name="FAISS_Vector_Retrieval", run_type="retriever")
def recuperar_contexto(pregunta, top_k=3):
    q_emb = obtener_embeddings([pregunta])
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, top_k)
    return [{**CHUNKS[i], "score": float(s)} for s, i in zip(scores[0], idxs[0]) if i != -1]

In [ ]:
# Prompt del sistema con instrucciones estrictas para evitar alucinaciones
SYSTEM_PROMPT = """Eres el asistente tecnico de compatibilidad de SP Digital.
1. Responde SOLO usando la informacion en <contexto>.
2. Si falta el dato, indica que no puedes confirmar y deriva a un vendedor humano.
3. Nunca inventes sockets, wattajes, chipsets ni modelos que no aparezcan
   textualmente en <contexto>. Esto incluye la seccion de Recomendacion:
   si vas a sugerir un producto o especificacion alternativa, esta debe
   estar en <contexto>; si no lo esta, di explicitamente que no tienes
   alternativas disponibles en el catalogo consultado y deriva a un
   vendedor humano para evaluar opciones fuera del catalogo.
4. Responde en 3 partes: Veredicto / Justificacion / Recomendacion.
5. Tono tecnico y breve (max. 120 palabras)."""

def construir_prompt(pregunta, fragmentos):
    contexto = "\n\n".join(f"[{f['metadata']['sku']}] {f['texto']}" for f in fragmentos)
    return f'<contexto>\n{contexto}\n</contexto>\n\nPregunta: "{pregunta}"'

In [ ]:
# Función auxiliar para hacer consultas de prueba rápidas por consola
def preguntar(pregunta):
    fragmentos = recuperar_contexto(pregunta)
    prompt = construir_prompt(pregunta, fragmentos)
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2, max_tokens=250,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": prompt}],
    )
    print("CONTEXTO USADO:", [f["metadata"]["sku"] for f in fragmentos])
    print("\nRESPUESTA:\n", resp.choices[0].message.content)

In [ ]:
preguntar("Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?")

CONTEXTO USADO: ['MB-B450-MSI', 'GPU-4070-ASUS', 'NOTA-PCIE-RETROCOMPAT']

RESPUESTA:
 **Veredicto:** **Sí, compatible.**

**Justificación:** La GPU ASUS RTX 4070 (PCIe 4.0 x16) funciona en el slot PCIe 3.0 x16 de la MSI B450 Tomahawk Max. La retrocompatibilidad PCIe garantiza operación sin problemas, aunque el ancho de banda teórico se limita a PCIe 3.0 (sin impacto práctico en rendimiento gamer).

**Recomendación:** Verifica que tu fuente de alimentación sea **≥650W** (recomendado por ASUS). No hay alternativas en el catálogo para esta combinación.


In [ ]:
preguntar("Que fuente de poder necesito para la RTX 4070 si mi fuente actual es de 450W?")

CONTEXTO USADO: ['GPU-4070-ASUS', 'PSU-EVGA-450', 'PSU-CORSAIR-650']

RESPUESTA:
 **Veredicto:** **Incompatibilidad crítica.**
La **RTX 4070 ASUS** requiere **12VHPWR** (1x 16-pin) y **mínimo 650W**. Tu fuente actual (**EVGA 450W**) no cumple con **ninguno** de estos requisitos.

**Justificación:**
- **Potencia:** 450W es **insuficiente** (mínimo exigido: 650W).
- **Conector:** Falta el **12VHPWR** (solo tiene PCIe 8-pin).
- **Riesgo:** Overclocking forzado, inestabilidad o daño a componentes.

**Recomendación:**
Usa la **Corsair RM650x (650W, 80+ Gold)**. Cumple con **potencia, eficiencia y conector 12VHPWR** (con adaptador incluido). *No hay alternativas en el catálogo.* Deriva a vendedor humano para evaluar opciones fuera de stock.


In [ ]:
# Módulo para cruzar los SKUs recuperados contra las reglas del negocio
def validar_contra_reglas(skus_en_contexto):
    for (a, b), regla in REGLAS_COMPATIBILIDAD.items():
        if a in skus_en_contexto and b in skus_en_contexto:
            return regla
    return None

# Probamos con un par que sí esté registrado
resultado = validar_contra_reglas(["GPU-4070-ASUS", "MB-B450-MSI"])
print("Regla encontrada:", resultado)

{'compatible': True, 'motivo': 'El slot PCIe 3.0 x16 acepta la GPU (PCIe 4.0 es retrocompatible).', 'requisito_adicional': 'Fuente >= 650W con conector 12VHPWR o adaptador.'}


In [ ]:
# Pipeline principal RAG registrado en LangSmith para ver toda la trazabilidad
@traceable(name="Pipeline_RAG_SP_Digital", run_type="chain")
def ejecutar_pipeline_rag(pregunta):
    fragmentos = recuperar_contexto(pregunta, top_k=3)
    prompt = construir_prompt(pregunta, fragmentos)
    
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.2,
        max_tokens=250,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
    )
    
    respuesta = resp.choices[0].message.content
    skus = [f["metadata"]["sku"] for f in fragmentos]
    regla = validar_contra_reglas(skus)
    
    return {
        "pregunta": pregunta,
        "respuesta_llm": respuesta,
        "skus_recuperados": skus,
        "regla_aplicada": regla,
        "fragmentos": fragmentos
    }

In [ ]:
# Probamos el pipeline completo con una pregunta de prueba
resultado = ejecutar_pipeline_rag("Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?")

print("Pregunta:", resultado["pregunta"])
print("-" * 60)
print("Contexto (SKUs recuperados):", resultado["skus_recuperados"])
print("-" * 60)
print("Respuesta del modelo:\n", resultado["respuesta_llm"])
print("-" * 60)
print("Validación de regla:", resultado["regla_aplicada"])

PREGUNTA: Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?
------------------------------------------------------------
SKUs RECUPERADOS (FAISS): ['MB-B450-MSI', 'GPU-4070-ASUS', 'NOTA-PCIE-RETROCOMPAT']
------------------------------------------------------------
RESPUESTA DEL LLM:
 **Veredicto:** ✅ **Sí, compatible.**

**Justificación:** La **MSI B450 Tomahawk Max** cuenta con un **slot PCIe 3.0 x16**, mientras que la **ASUS RTX 4070** es una tarjeta **PCIe 4.0 x16**. Según la nota técnica de retrocompatibilidad PCIe, la tarjeta funcionará sin problemas, aunque el ancho de banda se limitará a **PCIe 3.0** (sin impacto práctico en rendimiento gamer).

**Recomendación:**
- Verifica que tu **fuente de poder** sea **≥650W** (recomendación de ASUS).
- Confirma que el **conector 12VHPWR** de la GPU está presente en la placa (común en B450).
- *No hay alternativas en el catálogo para PCIe 4.0 en esta placa.*
------------------------------------------------------------
REGL

In [ ]:
import re

# Casos de prueba para evaluar la precisión del sistema frente al LLM directo
SET_PRUEBA = [
    {
        "pregunta": "Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?",
        "veredicto_esperado": "COMPATIBLE"
    },
    {
        "pregunta": "Puedo instalar un Ryzen 7 7800X3D en mi placa MSI B450 Tomahawk Max?",
        "veredicto_esperado": "NO COMPATIBLE"
    },
    {
        "pregunta": "Sirve la memoria RAM Kingston DDR5 de 32GB para la placa MSI B450?",
        "veredicto_esperado": "NO COMPATIBLE"
    },
    {
        "pregunta": "Puedo levantar la RTX 4070 con la fuente EVGA de 450W?",
        "veredicto_esperado": "NO COMPATIBLE"
    },
    {
        "pregunta": "Tienen disponibilidad de un cooler liquido de 360mm para gabinete ITX?",
        "veredicto_esperado": "SIN REGLA -> DERIVAR"
    }
]

def extraer_veredicto_llm(texto_respuesta):
    m = re.search(r"Veredicto:\**\s*✅?\s*\**\s*([^\n*]+)", texto_respuesta, re.IGNORECASE)
    texto = m.group(1).strip() if m else texto_respuesta[:40]
    if "no" in texto.lower() and "compat" in texto.lower():
        return "NO COMPATIBLE"
    if "compat" in texto.lower():
        return "COMPATIBLE"
    return "NO DETECTADO"

def evaluar_caso(pregunta):
    fragmentos = recuperar_contexto(pregunta, top_k=3)
    prompt = construir_prompt(pregunta, fragmentos)
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2, max_tokens=250,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": prompt}],
    )
    respuesta_llm = resp.choices[0].message.content
    skus = [f["metadata"]["sku"] for f in fragmentos]
    regla = validar_contra_reglas(skus)

    veredicto_llm = extraer_veredicto_llm(respuesta_llm)
    veredicto_sistema = ("COMPATIBLE" if regla["compatible"] else "NO COMPATIBLE") if regla else "SIN REGLA -> DERIVAR"

    return veredicto_llm, veredicto_sistema

aciertos_sistema = 0
coincidencias_llm = 0
for caso in SET_PRUEBA:
    veredicto_llm, veredicto_sistema = evaluar_caso(caso["pregunta"])
    ok_sistema = veredicto_sistema == caso["veredicto_esperado"]
    ok_llm_vs_sistema = veredicto_llm == veredicto_sistema
    aciertos_sistema += int(ok_sistema)
    coincidencias_llm += int(ok_llm_vs_sistema)

    print(f"Pregunta: {caso['pregunta']}")
    print(f"  Veredicto del LLM sin filtro : {veredicto_llm}")
    print(f"  Veredicto final del sistema  : {veredicto_sistema}  {'[OK]' if ok_sistema else '[FALLA]'} (Esperado: {caso['veredicto_esperado']})")
    if not ok_llm_vs_sistema:
        print(f"  -> El validador por regla corrigió al LLM (El LLM decía '{veredicto_llm}')")
    print()

print(f"Precisión del sistema final con validador: {aciertos_sistema}/{len(SET_PRUEBA)} ({100*aciertos_sistema/len(SET_PRUEBA):.0f}%)")
print(f"Coincidencia LLM directo vs sistema final: {coincidencias_llm}/{len(SET_PRUEBA)} ({100*coincidencias_llm/len(SET_PRUEBA):.0f}%)")

Pregunta: Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?
  Veredicto crudo del LLM   : COMPATIBLE
  Veredicto final (sistema) : COMPATIBLE  OK (esperado: COMPATIBLE)

Pregunta: Puedo instalar un Ryzen 7 7800X3D en mi placa MSI B450 Tomahawk Max?
  Veredicto crudo del LLM   : NO COMPATIBLE
  Veredicto final (sistema) : NO COMPATIBLE  OK (esperado: NO COMPATIBLE)

Pregunta: Sirve la memoria RAM Kingston DDR5 de 32GB para la placa MSI B450?
  Veredicto crudo del LLM   : NO COMPATIBLE
  Veredicto final (sistema) : NO COMPATIBLE  OK (esperado: NO COMPATIBLE)

Pregunta: Puedo levantar la RTX 4070 con la fuente EVGA de 450W?
  Veredicto crudo del LLM   : NO DETECTADO
  Veredicto final (sistema) : NO COMPATIBLE  OK (esperado: NO COMPATIBLE)
  -> El validador corrigio al LLM (LLM decia 'NO DETECTADO')

Pregunta: Tienen disponibilidad de un cooler liquido de 360mm para gabinete ITX?
  Veredicto crudo del LLM   : NO COMPATIBLE
  Veredicto final (sistema) : SIN REGLA -> DERIVAR

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

# Dataset de prueba con las respuestas esperadas (Ground Truth) para la evaluación de RAG
DATASET_EVALUACION = [
    {
        "inputs": {"question": "Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?"},
        "outputs": {
            "expected_verdict": "COMPATIBLE",
            "expected_skus": ["MB-B450-MSI", "GPU-4070-ASUS"]
        }
    },
    {
        "inputs": {"question": "Puedo instalar un Ryzen 7 7800X3D en mi placa MSI B450 Tomahawk Max?"},
        "outputs": {
            "expected_verdict": "NO COMPATIBLE",
            "expected_skus": ["MB-B450-MSI", "CPU-R7-7800X3D"]
        }
    },
    {
        "inputs": {"question": "Sirve la memoria RAM Kingston DDR5 de 32GB para la placa MSI B450?"},
        "outputs": {
            "expected_verdict": "NO COMPATIBLE",
            "expected_skus": ["MB-B450-MSI", "RAM-DDR5-32GB-KINGSTON"]
        }
    },
    {
        "inputs": {"question": "Puedo levantar la RTX 4070 con la fuente EVGA de 450W?"},
        "outputs": {
            "expected_verdict": "NO COMPATIBLE",
            "expected_skus": ["GPU-4070-ASUS", "PSU-EVGA-450"]
        }
    },
    {
        "inputs": {"question": "Tienen disponibilidad de un cooler liquido de 360mm para gabinete ITX?"},
        "outputs": {
            "expected_verdict": "SIN REGLA -> DERIVAR",
            "expected_skus": []
        }
    }
]

# Evaluador 1: Context Recall (revisa si FAISS recuperó los SKUs necesarios)
def eval_context_recall(run, example):
    skus_obtenidos = run.outputs.get("skus_recuperados", [])
    skus_esperados = example.outputs.get("expected_skus", [])
    
    if not skus_esperados:
        score = 1.0
    else:
        coincidencias = set(skus_esperados).intersection(set(skus_obtenidos))
        score = len(coincidencias) / len(skus_esperados)
        
    return {"key": "rag_context_recall", "score": score}

# Evaluador 2: Faithfulness (usamos el modelo como juez para ver si la respuesta es fiel al contexto)
def eval_faithfulness_llm_judge(run, example):
    respuesta = run.outputs.get("respuesta_llm", "")
    fragmentos = run.outputs.get("fragmentos", [])
    contexto_str = "\n".join([f["texto"] for f in fragmentos])
    
    judge_prompt = f"""
    Eres un auditor técnico estricto. Evalúa si la siguiente 'Respuesta' está respaldada 100% por la información en 'Contexto'.
    Si la respuesta menciona datos, sockets, watts o compatibilidades que NO aparecen en 'Contexto', responde 'INFIEL'.
    Si la respuesta deriva a un humano porque falta información, responde 'FIEL'.
    Si todo está respaldado en el contexto, responde 'FIEL'.

    Contexto: {contexto_str}
    Respuesta: {respuesta}

    Resultado (responde SOLO la palabra FIEL o INFIEL):
    """
    
    judge_resp = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.0,
        messages=[{"role": "user", "content": judge_prompt}]
    )
    
    resultado = judge_resp.choices[0].message.content.strip().upper()
    score = 1.0 if "FIEL" in resultado else 0.0
    return {"key": "rag_faithfulness", "score": score}

# Evaluador 3: Accuracy del veredicto final
def eval_verdict_accuracy(run, example):
    regla = run.outputs.get("regla_aplicada")
    if regla:
        veredicto_sistema = "COMPATIBLE" if regla["compatible"] else "NO COMPATIBLE"
    else:
        veredicto_sistema = "SIN REGLA -> DERIVAR"
        
    esperado = example.outputs.get("expected_verdict")
    return {"key": "verdict_accuracy", "score": 1.0 if veredicto_sistema == esperado else 0.0}

def predict_rag(inputs):
    res = ejecutar_pipeline_rag(inputs["question"])
    return res

print("=== INICIANDO EVALUACIÓN RAG Y LANGSMITH ===")

try:
    ls_client = Client()
    dataset_name = "SP-Digital-RAG-Benchmark"
    
    if not ls_client.has_dataset(dataset_name=dataset_name):
        dataset = ls_client.create_dataset(dataset_name=dataset_name, description="Casos de prueba de compatibilidad RAG")
        for item in DATASET_EVALUACION:
            ls_client.create_example(inputs=item["inputs"], outputs=item["outputs"], dataset_id=dataset.id)
    
    # Corremos el experimento de evaluación en LangSmith
    results = evaluate(
        predict_rag,
        data=dataset_name,
        evaluators=[eval_context_recall, eval_faithfulness_llm_judge, eval_verdict_accuracy],
        experiment_prefix="eval-mistral-rag",
    )
    print("\nEvaluación de LangSmith completada. Revisa los resultados en el dashboard web.")

except Exception as e:
    print(f"\nNo se pudo conectar con LangSmith ({e}). Corriendo evaluación local:\n")
    
    for item in DATASET_EVALUACION:
        q = item["inputs"]["question"]
        out = predict_rag({"question": q})
        
        class DummyObj:
            def __init__(self, d): self.outputs = d
            
        run_dummy = DummyObj(out)
        ex_dummy = DummyObj(item["outputs"])
        
        recall = eval_context_recall(run_dummy, ex_dummy)
        faith = eval_faithfulness_llm_judge(run_dummy, ex_dummy)
        acc = eval_verdict_accuracy(run_dummy, ex_dummy)
        
        print(f"Pregunta: {q}")
        print(f"  - Context Recall  : {recall['score'] * 100:.0f}%")
        print(f"  - LLM Faithfulness: {faith['score'] * 100:.0f}%")
        print(f"  - Accuracy        : {acc['score'] * 100:.0f}%\n")

=== INICIANDO EVALUACIÓN RAG Y LANGSMITH ===
View the evaluation results for experiment: 'eval-mistral-rag-2b141eae' at:
https://smith.langchain.com/o/a70ff840-d2cc-4d52-8d5e-b935b4fbc9b0/datasets/c030cc5f-c756-4685-a21a-300c5146bbef/compare?selectedSessions=a9cb232f-f3d7-407a-b46e-caca59cf1f45



✅ Evaluación de LangSmith completada exitosamente. Revisa el dashboard en smith.langchain.com
